In [2]:
import torch
from thop import profile, clever_format
from lib.moco.mocolsk_net import MoCoLSKNet
from lib.models import denosing_unet, SelfAttention

# on the landsat_cn20 dataset

In [ ]:
batch_size = 1
image_size = 160

lst_lr_itp = torch.randn(batch_size, 1, image_size, image_size)
ref_hr = torch.randn(batch_size, 6, image_size, image_size)
ndxi_hr = torch.randn(batch_size, 3, image_size, image_size)
dem_hr = torch.randn(batch_size, 1, image_size, image_size)
lulc_hr = torch.randint(0, 23, (batch_size, 1, image_size, image_size))
mask = torch.ones(batch_size, 11)
xt = torch.randn(batch_size, 1, image_size, image_size)
t = torch.randint(0, 15, (batch_size,))
guidance = torch.randn(batch_size, 10, image_size, image_size)

In [6]:
factor = 20
lst_lr = torch.randn(batch_size, 1, image_size//factor, image_size//factor)
# stage is more important than n_resblocks
model_moco = MoCoLSKNet(num_feats=32, scale=factor, n_resblocks=2, num_stages=2, reduction=16)
flops_moco, params_moco = profile(model_moco, inputs=(lst_lr,guidance))
flops_moco, params_moco = clever_format([flops_moco, params_moco], "%.2f")
print(params_moco, flops_moco)

[INFO] Register count_relu() for <class 'torch.nn.modules.activation.LeakyReLU'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.ConvTranspose2d'>.
[INFO] Register count_prelu() for <class 'torch.nn.modules.activation.PReLU'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
41.29M 556.25G


In [42]:
model_pgdm = denosing_unet(ch_base=8, attn_levels=[2,3], num_affine=3)
n_timestep = 3
flops_pgdm, params_pgdm = profile(model_pgdm, inputs=(lst_lr_itp, ref_hr, ndxi_hr, dem_hr, lulc_hr, mask, xt, t))
flops_pgdm *= n_timestep
flops_pgdm, params_pgdm = clever_format([flops_pgdm, params_pgdm], "%.2f")
print(params_pgdm, flops_pgdm)

[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pixelshuffle.PixelShuffle'>.
892.54K 3.70G


# on the GrokLST dataset

In [3]:
batch_size = 1
image_size = 512

lst_lr_itp = torch.randn(batch_size, 1, image_size, image_size)
ref_hr = torch.randn(batch_size, 6, image_size, image_size)
ndxi_hr = torch.randn(batch_size, 3, image_size, image_size)
dem_hr = torch.randn(batch_size, 1, image_size, image_size)
lulc_hr = torch.randint(0, 23, (batch_size, 1, image_size, image_size))
mask = torch.ones(batch_size, 11)
xt = torch.randn(batch_size, 1, image_size, image_size)
t = torch.randint(0, 15, (batch_size,))
guidance = torch.randn(batch_size, 10, image_size, image_size)

In [6]:
model_pgdm = denosing_unet(ch_base=32, attn_levels=[3,4], ch_mult=[1,2,4,8,16], num_affine=3)
n_timestep = 3
flops_pgdm, params_pgdm = profile(model_pgdm, inputs=(lst_lr_itp, ref_hr, ndxi_hr, dem_hr, lulc_hr, mask, xt, t))
flops_pgdm *= n_timestep
flops_pgdm, params_pgdm = clever_format([flops_pgdm, params_pgdm], "%.2f")
print(params_pgdm, flops_pgdm)

[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pixelshuffle.PixelShuffle'>.
48.09M 765.62G


In [7]:
factor = 8
lst_lr = torch.randn(batch_size, 1, image_size//factor, image_size//factor)
# stage is more important than n_resblocks
model_moco = MoCoLSKNet(scale=factor, n_resblocks=4, num_stages=4)
flops_moco, params_moco = profile(model_moco, inputs=(lst_lr,guidance))
flops_moco, params_moco = clever_format([flops_moco, params_moco], "%.2f")
print(params_moco, flops_moco)

[INFO] Register count_relu() for <class 'torch.nn.modules.activation.LeakyReLU'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.ConvTranspose2d'>.
[INFO] Register count_prelu() for <class 'torch.nn.modules.activation.PReLU'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.


d:\Anaconda3\envs\new\lib\site-packages\thop\vision\calc_func.py:53: UserWarning: This API is being deprecated
  warnings.warn("This API is being deprecated")


40.91M 6.02T
